Chargement des données

In [1]:
import pandas as pd

from Modules.text_processor import TextProcessor

# readind the dataset
file_path = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-02-25/article_dat.csv'
df = pd.read_csv(file_path)

# instantiating the text processor
text_processor = TextProcessor(download=True)

[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /home/onyxia/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /home/onyxia/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
df.sample(5)

> Néttoyage agressif
tout le texte en minuscule
On retire la pontuation
Remplacement des espaces multiples par des espaces simple
Rétrait des mot très fréquents sans valeur : of, the, and
Suppression des espaces en fin et debut de texte

In [39]:
text_1 = df['title'][0]
print(text_1)
" ".join(text_processor.preprocess(text="")) == ""

Desired Sterilization Procedure at the Time of Cesarean Delivery According to Insurance Status.


True

## Traitements initiaux

- Verifier les `Nan`, les `""`, etc ... dans les variables de textes (`titre`, `study_type`, `abstract`, `study_aim` etc...)
- ....
- ....
- ...

In [15]:
df.columns

Index(['pmid', 'doi', 'jabbrv', 'journal', 'year', 'month', 'day', 'title',
       'abstract', 'keywords', 'study_aim', 'study_location',
       'study_year_start', 'study_year_end', 'study_type', 'data_source',
       'race1', 'race1_ss', 'race2', 'race2_ss', 'race3', 'race3_ss', 'race4',
       'race4_ss', 'race5', 'race5_ss', 'race6', 'race6_ss', 'race7',
       'race7_ss', 'race8', 'race8_ss', 'eth1', 'eth1_ss', 'eth2', 'eth2_ss',
       'eth3', 'eth3_ss', 'eth4', 'eth4_ss', 'eth5', 'eth5_ss', 'eth6',
       'eth6_ss', 'eth7', 'eth7_ss', 'eth8', 'eth8_ss', 'access_to_care',
       'treatment_received', 'health_outcome', 'cancer_ovarian',
       'cancer_uterine', 'cancer_cervical', 'cancer_vulvar', 'other_gyn_onc',
       'endo', 'fibroids', 'other_gyn_surg', 'fert', 'matmorbmort',
       'other_preg', 'phys_div', 'other', 'covid'],
      dtype='str')

In [21]:
from Modules.base_info_class import REQUIRED_FIELDS

print(REQUIRED_FIELDS)
# Selection des colonnes d'intérêt
target_col = [x for x in REQUIRED_FIELDS if x in df.columns]
target_col

{'data_source', 'authors', 'study_type', 'abstract', 'title', 'study_year_start', 'study_aim', 'study_location', 'study_year_end'}


['data_source',
 'study_type',
 'abstract',
 'title',
 'study_year_start',
 'study_aim',
 'study_location',
 'study_year_end']

In [45]:
# Selection de la table d'intérêt
df_target = df.loc[:, target_col]
df_target.sample(4, random_state=4)

,data_source,study_type,abstract,title,study_year_start,study_aim,study_location,study_year_end
99,National Cancer Database,Retrospective cohort,Delays in time to treatment initiation (TTI) w...,Delays in definitive cervical cancer treatment...,2004.0,to investigate the disparities in delays in ti...,USA,2014.0
248,American College of Surgeons' National Surgica...,Retrospective Cohort,Black race has been associated with increased ...,Laparoscopy decreases the disparity in postope...,2010.0,Black race has been associated with increased ...,USA,2015.0
92,"walter reed army medical center, wilford hall ...",Retrospective cohort,To evaluate assisted reproduction technology (...,Will decreasing assisted reproduction technolo...,2000.0,To evaluate ART utilization and outcomes in mi...,NaN,2005.0
61,"Surveillance, Epidemiology, and End Results; M...",Retrospective cohort,To examine whether treatment with guideline-re...,Racial disparities in the treatment of advance...,1995.0,To examine whether treatment with guideline-re...,USA,2007.0


> Affichage des types d'articles que compte la base de données

In [48]:
print(df["study_type"].unique().tolist())
print(len(df["study_type"].unique().tolist()))

['Retrospective cohort', 'Cross-sectional', 'Case-control', 'Prospective cohort', 'RCT', 'Registry', 'Retrospective Cohort', 'Prospective Cohort', 'Cross-Sectional', nan]
10


On observe que plusieurs modalités renvoie au même type d'articles seulement avec quelques caractères qui diffèrent. On va donc appliquer une traitement sur les colonnes de cette table d'intérêt

In [49]:
# Application d'un premier traitement au colonne de cette table
for col in target_col:
    df_target[col] = df_target[col].apply(
        lambda x: None if " ".join(text_processor.preprocess(x)) == "" else " ".join(text_processor.preprocess(x)) # Reconstitution des phrases apres avoir réalisé le prétraitement sur les chaines de caractères
        # On s'assure de bien concerver NA lorsque l'information n'est pas disponible
    )

df_target.sample(4, random_state=4)

,data_source,study_type,abstract,title,study_year_start,study_aim,study_location,study_year_end
99,national cancer database,retrospective cohort,delays time treatment initiation tti definitiv...,delays definitive cervical cancer treatment an...,None,investigate disparities delays time treatment ...,usa,None
248,american college surgeons national surgical qu...,retrospective cohort,black race associated increased day morbidity ...,laparoscopy decreases disparity postoperative ...,None,black race associated increased day morbidity ...,usa,None
92,walter reed army medical center wilford hall m...,retrospective cohort,evaluate assisted reproduction technology art ...,decreasing assisted reproduction technology co...,None,evaluate art utilization outcomes minority wom...,NaN,None
61,surveillance epidemiology end results medicare...,retrospective cohort,examine whether treatment guideline recommende...,racial disparities treatment advanced epitheli...,None,examine whether treatment guideline recommende...,usa,None


> Vérification du prétraitement en afficahant le type des études

In [50]:
print(df_target["study_type"].unique().tolist())
print(len(df_target["study_type"].unique().tolist()))

['retrospective cohort', 'cross sectional', 'case control', 'prospective cohort', 'rct', 'registry', nan]
7
